In [1]:
import cv2
from ultralytics import YOLO

# Load the model
yolo = YOLO('yolo_model_trined_on_head.pt')
yolo.model.names

# Load the video capture

{0: 'Human head'}

In [2]:
# Function to get class colors
def getColours(cls_num):
    base_colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]
    color_index = cls_num % len(base_colors)
    increments = [(1, -2, 1), (-2, 1, -1), (1, -1, 2)]
    color = [base_colors[color_index][i] + increments[color_index][i] * 
    (cls_num // len(base_colors)) % 256 for i in range(3)]
    return tuple(color)

In [5]:
import torch
import torchvision
print(torch.__version__, torchvision.__version__)

2.6.0+cu118 0.21.0+cu118


In [1]:
videoCap = cv2.VideoCapture(0)
while True:
    ret, frame = videoCap.read()
    if not ret:
        continue
    try:
        results = yolo.track(frame, stream=True)
    except Exception as e:
        print(f"Error in YOLO tracking: {e}")
    # results = yolo.track(frame, stream=True)
    for result in results:
        # get the classes names
        classes_names = result.names

        # iterate over each box
        for box in result.boxes:
            # check if confidence is greater than 40 percent
            if box.conf[0] > 0.4:
                # get coordinates
                [x1, y1, x2, y2] = box.xyxy[0]
                # convert to int
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # get the class
                cls = int(box.cls[0])

                # get the class name
                class_name = classes_names[cls]

                # get the respective colour
                colour = getColours(cls)

                # draw the rectangle
                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)

                # put the class name and confidence on the image
                cv2.putText(frame, f'{classes_names[int(box.cls[0])]} {box.conf[0]:.2f}', (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 1, colour, 2)             
    # show the image
    cv2.imshow('frame', frame)
    # break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
# release the video capture and destroy all windows
videoCap.release()
cv2.destroyAllWindows()



NameError: name 'cv2' is not defined

On Shooted Video 

In [4]:
import cv2

video_path = "E:/manorath Leriya/ras/1/VTS_02_2.VOB"  # Replace with your actual video path
videoCap = cv2.VideoCapture(video_path)

while True:
    ret, frame = videoCap.read()
    if not ret:
        break  # Exit loop when the video ends

    try:
        results = yolo.track(frame, stream=True,device='cuda')
    except Exception as e:
        print(f"Error in YOLO tracking: {e}")
        continue

    for result in results:
        classes_names = result.names

        for box in result.boxes:
            if box.conf[0] > 0.4:
                [x1, y1, x2, y2] = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                cls = int(box.cls[0])
                class_name = classes_names[cls]
                colour = getColours(cls)

                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
                cv2.putText(frame, f'{class_name} {box.conf[0]:.2f}', (x1, y1),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, colour, 2)

    cv2.imshow('frame', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

videoCap.release()
cv2.destroyAllWindows()



0: 512x640 7 Human heads, 88.2ms
Speed: 5.7ms preprocess, 88.2ms inference, 4.7ms postprocess per image at shape (1, 3, 512, 640)

0: 512x640 1 Human head, 27.3ms
Speed: 4.0ms preprocess, 27.3ms inference, 1.6ms postprocess per image at shape (1, 3, 512, 640)

0: 512x640 2 Human heads, 7.0ms
Speed: 1.6ms preprocess, 7.0ms inference, 1.2ms postprocess per image at shape (1, 3, 512, 640)

0: 512x640 6 Human heads, 8.0ms
Speed: 1.6ms preprocess, 8.0ms inference, 1.4ms postprocess per image at shape (1, 3, 512, 640)

0: 512x640 11 Human heads, 7.3ms
Speed: 1.6ms preprocess, 7.3ms inference, 2.1ms postprocess per image at shape (1, 3, 512, 640)

0: 512x640 5 Human heads, 7.4ms
Speed: 1.5ms preprocess, 7.4ms inference, 1.4ms postprocess per image at shape (1, 3, 512, 640)

0: 512x640 9 Human heads, 7.5ms
Speed: 1.5ms preprocess, 7.5ms inference, 1.2ms postprocess per image at shape (1, 3, 512, 640)

0: 512x640 8 Human heads, 13.1ms
Speed: 1.9ms preprocess, 13.1ms inference, 2.5ms postproces

In [17]:
import cv2

# video_path = "path/to/your/video.mp4"  # Replace with your actual video path
videoCap = cv2.VideoCapture(video_path)

# Get total number of frames in the video
total_frames = int(videoCap.get(cv2.CAP_PROP_FRAME_COUNT))

# Callback function for trackbar (updates the frame position)
def on_trackbar(val):
    videoCap.set(cv2.CAP_PROP_POS_FRAMES, val)

# Create OpenCV window and trackbar
cv2.namedWindow("Video")
cv2.createTrackbar("Seek", "Video", 0, total_frames - 1, on_trackbar)

while True:
    current_frame = int(videoCap.get(cv2.CAP_PROP_POS_FRAMES))  # Get current frame position
    ret, frame = videoCap.read()
    if not ret:
        break  # Exit when video ends

    try:
        results = yolo.track(frame, stream=True)
    except Exception as e:
        print(f"Error in YOLO tracking: {e}")
        continue

    for result in results:
        classes_names = result.names

        for box in result.boxes:
            if box.conf[0] > 0.4:
                [x1, y1, x2, y2] = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                cls = int(box.cls[0])
                class_name = classes_names[cls]
                colour = getColours(cls)

                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
                cv2.putText(frame, f'{class_name} {box.conf[0]:.2f}', (x1, y1),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, colour, 2)

    # Update trackbar position
    cv2.setTrackbarPos("Seek", "Video", current_frame)

    cv2.imshow("Video", frame)

    # key = cv2.waitKey(30)  # Delay for smooth playback

    # if key == ord('q'):
    #     break  # Quit when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

videoCap.release()
cv2.destroyAllWindows()


With SeekBar

In [ ]:
import cv2
video_path = "E:/manorath Leriya/ras/1/VTS_02_2.VOB" 
# video_path = "path/to/your/video.mp4"  # Replace with your actual video path
videoCap = cv2.VideoCapture(video_path)

# Get total number of frames in the video
total_frames = int(videoCap.get(cv2.CAP_PROP_FRAME_COUNT))

# Variable to store trackbar position
trackbar_pos = 0
is_trackbar_moving = False  # To track if user is manually moving trackbar

# Callback function for trackbar movement
def on_trackbar(val):
    global is_trackbar_moving
    is_trackbar_moving = True  # Trackbar was moved manually
    videoCap.set(cv2.CAP_PROP_POS_FRAMES, val)

# Create OpenCV window and trackbar
cv2.namedWindow("Video")
cv2.createTrackbar("Seek", "Video", 0, total_frames - 1, on_trackbar)

while True:
    if not is_trackbar_moving:
        current_frame = int(videoCap.get(cv2.CAP_PROP_POS_FRAMES))  # Get current frame position
    else:
        current_frame = cv2.getTrackbarPos("Seek", "Video")  # Get frame from trackbar

    ret, frame = videoCap.read()
    if not ret:
        break  # Exit when video ends

    try:
        results = yolo.track(frame, stream=True , device='cuda')    
    except Exception as e:
        print(f"Error in YOLO tracking: {e}")
        continue

    for result in results:
        classes_names = result.names

        for box in result.boxes:
            if box.conf[0] > 0.4:
                [x1, y1, x2, y2] = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                cls = int(box.cls[0])
                class_name = classes_names[cls]
                colour = getColours(cls)

                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
                cv2.putText(frame, f'{class_name} {box.conf[0]:.2f}', (x1, y1),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, colour, 2)

    # Update trackbar position only when the user is not manually moving it
    if not is_trackbar_moving:
        cv2.setTrackbarPos("Seek", "Video", current_frame + 1)

    cv2.imshow("Video", frame)

    key = cv2.waitKey(30)  # Delay for smooth playback

    if key == ord('q'):
        break  # Quit when 'q' is pressed

    # Reset flag after manual movement
    is_trackbar_moving = False

videoCap.release()
cv2.destroyAllWindows()


Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'yolo' is not defined
Error in YOLO tracking: name 'y

KeyboardInterrupt: 

: 